# Day 1 — Amazon Reviews 2023: Chuẩn bị dữ liệu Next-Item Recommendation

**Mục tiêu cuối ngày**

1. Tạo thư mục dự án.
2. Tải dữ liệu `All_Beauty` và metadata chính thức.
3. Kiểm tra, làm sạch và ghép metadata.
4. Thực hiện iterative k-core filtering.
5. Chia dữ liệu theo thời gian thành train/validation/test.
6. Tạo file `.inter` để dùng với RecBole vào Ngày 2.
7. Lưu toàn bộ đầu ra lên Google Drive.

> Chạy từng ô từ trên xuống dưới. Không chọn **Run all** ngay lần đầu.

## Bước 1 — Kết nối Google Drive

Khi Google hỏi quyền truy cập, chọn tài khoản của bạn và bấm **Allow**.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Bước 2 — Tạo cấu trúc thư mục

Có thể đổi tên `ROOT` nếu bạn muốn đặt dự án ở vị trí khác trong Google Drive.

In [2]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/llm-next-item-recommendation")

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
RECBOLE_DIR = ROOT / "data" / "recbole" / "all_beauty"
OUTPUT_DIR = ROOT / "outputs"
NOTEBOOK_DIR = ROOT / "notebooks"

for folder in [RAW_DIR, PROCESSED_DIR, RECBOLE_DIR, OUTPUT_DIR, NOTEBOOK_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", ROOT)
print("Created folders:")
for folder in [RAW_DIR, PROCESSED_DIR, RECBOLE_DIR, OUTPUT_DIR, NOTEBOOK_DIR]:
    print("-", folder)

Project folder: /content/drive/MyDrive/llm-next-item-recommendation
Created folders:
- /content/drive/MyDrive/llm-next-item-recommendation/data/raw
- /content/drive/MyDrive/llm-next-item-recommendation/data/processed
- /content/drive/MyDrive/llm-next-item-recommendation/data/recbole/all_beauty
- /content/drive/MyDrive/llm-next-item-recommendation/outputs
- /content/drive/MyDrive/llm-next-item-recommendation/notebooks


## Bước 3 — Cài thư viện cần dùng trong Ngày 1

Ngày 1 chưa cần cài RecBole hoặc LLM.

In [3]:
!pip -q install -U pyarrow huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 39.9 MB/s eta 0:00:00


## Bước 4 — Import thư viện và cố định random seed

In [4]:
import sys
import json
import platform
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import pyarrow
import huggingface_hub

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("pyarrow:", pyarrow.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
pandas: 2.2.2
numpy: 2.0.2
pyarrow: 24.0.0
huggingface_hub: 1.21.0


## Bước 5 — Tải hai file chính thức từ Hugging Face

- Review/interactions: khoảng 139 MB.
- Product metadata: khoảng 60 MB.

Nếu file đã có trong cache, Hugging Face sẽ không tải lại toàn bộ.

In [6]:
from huggingface_hub import hf_hub_download

REPO_ID = "McAuley-Lab/Amazon-Reviews-2023"

# File lịch sử review/interactions
reviews_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    revision="fc4ce368192284e83719923a5ae3060ee90484a1",
    filename="raw_review_All_Beauty/full-00000-of-00001.parquet",
    local_dir=str(RAW_DIR),
)

# File thông tin sản phẩm
metadata_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    revision="14f1fc4a2c268267760379170656f09c6193e9bd",
    filename="raw_meta_All_Beauty/full-00000-of-00001.parquet",
    local_dir=str(RAW_DIR),
)

print("Reviews file:", reviews_path)
print("Metadata file:", metadata_path)

raw_review_All_Beauty/full-00000-of-0000(…):   0%|          | 0.00/139M [00:00<?, ?B/s]

raw_meta_All_Beauty/full-00000-of-00001.(…):   0%|          | 0.00/59.7M [00:00<?, ?B/s]

Reviews file: /content/drive/MyDrive/llm-next-item-recommendation/data/raw/raw_review_All_Beauty/full-00000-of-00001.parquet
Metadata file: /content/drive/MyDrive/llm-next-item-recommendation/data/raw/raw_meta_All_Beauty/full-00000-of-00001.parquet


## Bước 6 — Đọc dữ liệu

Chỉ đọc các cột cần thiết để tiết kiệm RAM.

In [7]:
review_columns = [
    "rating",
    "title",
    "text",
    "parent_asin",
    "user_id",
    "timestamp",
    "verified_purchase",
]

metadata_columns = [
    "parent_asin",
    "title",
    "features",
    "description",
    "average_rating",
    "rating_number",
    "price",
    "store",
    "main_category",
]

reviews = pd.read_parquet(reviews_path, columns=review_columns)
items = pd.read_parquet(metadata_path, columns=metadata_columns)

print("Reviews shape:", reviews.shape)
print("Metadata shape:", items.shape)

display(reviews.head(3))
display(items.head(3))

Reviews shape: (701528, 7)
Metadata shape: (112590, 9)


,rating,title,text,parent_asin,user_id,timestamp,verified_purchase
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,True
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,True
2,5.0,Yes!,"Smells good, feels great!",B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,True


,parent_asin,title,features,description,average_rating,rating_number,price,store,main_category
0,B01CUPMQZE,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",[],[],4.8,10,None,Howard Products,All Beauty
1,B076WQZGPM,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,[],[],4.5,3,None,Yes To,All Beauty
2,B000B658RI,Eye Patch Black Adult with Tie Band (6 Per Pack),[],[],4.4,26,None,Levine Health Products,All Beauty


## Bước 7 — Kiểm tra chất lượng dữ liệu ban đầu

In [9]:
def basic_audit(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows = []

    for col in df.columns:
        # Xử lý an toàn với cột chứa list, dict hoặc numpy array
        missing_count = df[col].apply(
            lambda x: (
                x is None
                or (
                    not isinstance(x, (list, tuple, dict, np.ndarray))
                    and pd.isna(x)
                )
            )
        ).sum()

        rows.append(
            {
                "dataset": name,
                "column": col,
                "dtype": str(df[col].dtype),
                "missing_count": int(missing_count),
                "missing_pct": round(
                    float(missing_count / len(df) * 100), 3
                ),
            }
        )

    return pd.DataFrame(rows)


# Tạo bảng kiểm tra dữ liệu
review_audit = basic_audit(reviews, "reviews")
item_audit = basic_audit(items, "items")


# Reviews không có cột dạng numpy array nên có thể kiểm tra toàn bộ dòng
duplicate_review_rows = reviews.duplicated().sum()

# Metadata có features và description dạng array,
# nên chỉ kiểm tra sản phẩm trùng theo mã parent_asin
duplicate_metadata_ids = items.duplicated(
    subset=["parent_asin"],
    keep="first"
).sum()


print("Exact duplicate review rows:", duplicate_review_rows)

print(
    "Duplicate metadata product IDs:",
    duplicate_metadata_ids
)

print(
    "Unique users:",
    reviews["user_id"].nunique()
)

print(
    "Unique parent products in reviews:",
    reviews["parent_asin"].nunique()
)

print(
    "Unique products in metadata:",
    items["parent_asin"].nunique()
)


display(review_audit)
display(item_audit)


# Gộp và lưu báo cáo audit
audit_table = pd.concat(
    [review_audit, item_audit],
    ignore_index=True
)

audit_table.to_csv(
    OUTPUT_DIR / "day1_data_audit.csv",
    index=False
)

print(
    "Audit file saved to:",
    OUTPUT_DIR / "day1_data_audit.csv"
)

Exact duplicate review rows: 7276
Duplicate metadata product IDs: 0
Unique users: 631986
Unique parent products in reviews: 112565
Unique products in metadata: 112590


,dataset,column,dtype,missing_count,missing_pct
0,reviews,rating,float64,0,0.0
1,reviews,title,object,0,0.0
2,reviews,text,object,0,0.0
3,reviews,parent_asin,object,0,0.0
4,reviews,user_id,object,0,0.0
5,reviews,timestamp,int64,0,0.0
6,reviews,verified_purchase,bool,0,0.0


,dataset,column,dtype,missing_count,missing_pct
0,items,parent_asin,object,0,0.000
1,items,title,object,0,0.000
2,items,features,object,0,0.000
3,items,description,object,0,0.000
4,items,average_rating,float64,0,0.000
5,items,rating_number,int64,0,0.000
6,items,price,object,0,0.000
7,items,store,object,11331,10.064
8,items,main_category,object,0,0.000


Audit file saved to: /content/drive/MyDrive/llm-next-item-recommendation/outputs/day1_data_audit.csv


## Bước 8 — Làm sạch interactions

Quy tắc:

- `parent_asin` được đổi thành `item_id`.
- Xóa dòng thiếu `user_id`, `item_id` hoặc `timestamp`.
- Chỉ xóa bản ghi trùng hoàn toàn theo user–item–timestamp.
- Giữ lại lượt tương tác lặp lại nếu thời điểm khác nhau.

In [10]:
interactions = reviews.rename(columns={"parent_asin": "item_id"}).copy()

interactions = interactions[
    ["user_id", "item_id", "timestamp", "rating", "verified_purchase"]
]

before = len(interactions)

interactions["timestamp"] = pd.to_numeric(
    interactions["timestamp"], errors="coerce"
)

interactions = interactions.dropna(
    subset=["user_id", "item_id", "timestamp"]
)

interactions = interactions[
    (interactions["user_id"].astype(str).str.len() > 0)
    & (interactions["item_id"].astype(str).str.len() > 0)
    & (interactions["timestamp"] > 0)
]

interactions = interactions.drop_duplicates(
    subset=["user_id", "item_id", "timestamp"]
)

interactions["user_id"] = interactions["user_id"].astype(str)
interactions["item_id"] = interactions["item_id"].astype(str)
interactions["timestamp"] = interactions["timestamp"].astype("int64")
interactions["datetime"] = pd.to_datetime(
    interactions["timestamp"], unit="ms", utc=True
)

interactions = interactions.sort_values(
    ["user_id", "timestamp", "item_id"]
).reset_index(drop=True)

print("Rows before cleaning:", before)
print("Rows after cleaning:", len(interactions))
print("Rows removed:", before - len(interactions))
print("Date range:", interactions["datetime"].min(), "to", interactions["datetime"].max())

display(interactions.head())

Rows before cleaning: 701528
Rows after cleaning: 694252
Rows removed: 7276
Date range: 2000-11-01 04:24:18+00:00 to 2023-09-09 00:39:36.666000+00:00


,user_id,item_id,timestamp,rating,verified_purchase,datetime
0,AE222BBOVZIF42YOOPNBXL4UUMYA,B013HR1A92,1457569672000,5.0,True,2016-03-10 00:27:52+00:00
1,AE222FP7YRNFCEQ2W3ZDIGMSYTLQ,B0BTT658PQ,1678158612143,5.0,True,2023-03-07 03:10:12.143000+00:00
2,AE222X475JC6ONXMIKZDFGQ7IAUA,B00PBDMRES,1483728924000,5.0,True,2017-01-06 18:55:24+00:00
3,AE222Y4WTST6BUZ4J5Y2H6QMBITQ,B00012FPSO,1372108302000,4.0,True,2013-06-24 21:11:42+00:00
4,AE2232TEZOEWQLAFEX2NA6VBGMYQ,B07QNPXBLH,1564465646197,5.0,True,2019-07-30 05:47:26.197000+00:00


## Bước 9 — Làm sạch product metadata và tạo `product_text`

In [11]:
def value_to_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return " ".join(
            str(x).strip() for x in value
            if x is not None and str(x).strip()
        )
    if isinstance(value, dict):
        return " ".join(
            f"{k}: {v}" for k, v in value.items()
            if v is not None and str(v).strip()
        )
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()

items_clean = items.rename(columns={"parent_asin": "item_id"}).copy()
items_clean = items_clean.dropna(subset=["item_id"])
items_clean["item_id"] = items_clean["item_id"].astype(str)
items_clean = items_clean.drop_duplicates(subset=["item_id"], keep="first")

for col in ["title", "features", "description"]:
    items_clean[f"{col}_text"] = items_clean[col].apply(value_to_text)

items_clean["product_text"] = (
    items_clean["title_text"]
    + ". "
    + items_clean["features_text"]
    + ". "
    + items_clean["description_text"]
)

items_clean["product_text"] = (
    items_clean["product_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip(" .")
)

items_clean = items_clean[
    [
        "item_id",
        "title_text",
        "features_text",
        "description_text",
        "product_text",
        "average_rating",
        "rating_number",
        "price",
        "store",
        "main_category",
    ]
]

print("Unique metadata items:", items_clean["item_id"].nunique())
print(
    "Items with non-empty product_text:",
    (items_clean["product_text"].str.len() > 0).sum(),
)

display(items_clean.head(3))

Unique metadata items: 112590
Items with non-empty product_text: 112579


,item_id,title_text,features_text,description_text,product_text,average_rating,rating_number,price,store,main_category
0,B01CUPMQZE,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",,,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,None,Howard Products,All Beauty
1,B076WQZGPM,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,,,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,None,Yes To,All Beauty
2,B000B658RI,Eye Patch Black Adult with Tie Band (6 Per Pack),,,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,None,Levine Health Products,All Beauty


## Bước 10 — Chỉ giữ interactions có metadata

LLM ở Ngày 3 cần title/features/description. Vì vậy, các item không có metadata sẽ không được dùng trong proof of concept.

In [12]:
before = len(interactions)

available_item_ids = set(items_clean["item_id"])
interactions = interactions[
    interactions["item_id"].isin(available_item_ids)
].copy()

print("Interactions before metadata matching:", before)
print("Interactions after metadata matching:", len(interactions))
print("Removed because metadata was unavailable:", before - len(interactions))

Interactions before metadata matching: 694252
Interactions after metadata matching: 694252
Removed because metadata was unavailable: 0


## Bước 11 — Chạy iterative 5-core filtering

**5-core** nghĩa là sau khi lọc hội tụ:

- Mỗi user còn lại có ít nhất 5 interactions.
- Mỗi item còn lại có ít nhất 5 interactions.

Không dừng chỉ sau một vòng vì khi xóa user ít tương tác, một số item có thể lại rơi xuống dưới ngưỡng và ngược lại.

In [13]:
def iterative_k_core(
    df: pd.DataFrame,
    user_k: int = 5,
    item_k: int = 5,
    verbose: bool = True,
) -> pd.DataFrame:
    result = df.copy()
    iteration = 0

    while True:
        iteration += 1
        before_rows = len(result)

        user_counts = result["user_id"].value_counts()
        valid_users = user_counts[user_counts >= user_k].index
        result = result[result["user_id"].isin(valid_users)]

        item_counts = result["item_id"].value_counts()
        valid_items = item_counts[item_counts >= item_k].index
        result = result[result["item_id"].isin(valid_items)]

        if verbose:
            print(
                f"Iteration {iteration}: "
                f"{len(result):,} rows | "
                f"{result['user_id'].nunique():,} users | "
                f"{result['item_id'].nunique():,} items"
            )

        if len(result) == before_rows or result.empty:
            break

    return result.reset_index(drop=True)

interactions_5core = iterative_k_core(
    interactions,
    user_k=5,
    item_k=5,
)

print("\nFinal 5-core statistics")
print("Interactions:", f"{len(interactions_5core):,}")
print("Users:", f"{interactions_5core['user_id'].nunique():,}")
print("Items:", f"{interactions_5core['item_id'].nunique():,}")

if not interactions_5core.empty:
    print(
        "Minimum interactions per user:",
        interactions_5core.groupby("user_id").size().min(),
    )
    print(
        "Minimum interactions per item:",
        interactions_5core.groupby("item_id").size().min(),
    )

Iteration 1: 5,512 rows | 899 users | 724 items
Iteration 2: 3,708 rows | 417 users | 510 items
Iteration 3: 3,154 rows | 334 users | 436 items
Iteration 4: 2,896 rows | 295 users | 403 items
Iteration 5: 2,782 rows | 279 users | 390 items
Iteration 6: 2,746 rows | 275 users | 385 items
Iteration 7: 2,734 rows | 273 users | 384 items
Iteration 8: 2,722 rows | 272 users | 382 items
Iteration 9: 2,687 rows | 268 users | 377 items
Iteration 10: 2,663 rows | 266 users | 373 items
Iteration 11: 2,639 rows | 264 users | 369 items
Iteration 12: 2,631 rows | 263 users | 368 items
Iteration 13: 2,627 rows | 262 users | 368 items
Iteration 14: 2,627 rows | 262 users | 368 items

Final 5-core statistics
Interactions: 2,627
Users: 262
Items: 368
Minimum interactions per user: 5
Minimum interactions per item: 5


In [21]:
core_results = []

for k in [5, 4, 3]:
    filtered_data = iterative_k_core(
        interactions,
        user_k=k,
        item_k=k,
        verbose=False
    )

    core_results.append({
        "k_core": f"{k}-core",
        "interactions": len(filtered_data),
        "users": filtered_data["user_id"].nunique(),
        "items": filtered_data["item_id"].nunique(),
        "min_interactions_per_user": (
            filtered_data.groupby("user_id").size().min()
            if not filtered_data.empty else 0
        ),
        "min_interactions_per_item": (
            filtered_data.groupby("item_id").size().min()
            if not filtered_data.empty else 0
        )
    })

core_comparison = pd.DataFrame(core_results)
display(core_comparison)

,k_core,interactions,users,items,min_interactions_per_user,min_interactions_per_item
0,5-core,2627,262,368,5,5
1,4-core,5628,596,855,4,4
2,3-core,9079,1266,1473,3,3


In [22]:
SELECTED_K = 3

selected_interactions = iterative_k_core(
    interactions,
    user_k=SELECTED_K,
    item_k=SELECTED_K,
    verbose=True
)

print("\nSelected dataset")
print("Core:", SELECTED_K)
print("Interactions:", len(selected_interactions))
print("Users:", selected_interactions["user_id"].nunique())
print("Items:", selected_interactions["item_id"].nunique())

print(
    "Minimum interactions per user:",
    selected_interactions.groupby("user_id").size().min()
)

print(
    "Minimum interactions per item:",
    selected_interactions.groupby("item_id").size().min()
)

Iteration 1: 17,151 rows | 5,961 users | 2,930 items
Iteration 2: 10,307 rows | 1,833 users | 1,698 items
Iteration 3: 9,444 rows | 1,409 users | 1,544 items
Iteration 4: 9,203 rows | 1,314 users | 1,492 items
Iteration 5: 9,138 rows | 1,286 users | 1,483 items
Iteration 6: 9,120 rows | 1,280 users | 1,480 items
Iteration 7: 9,106 rows | 1,275 users | 1,478 items
Iteration 8: 9,095 rows | 1,272 users | 1,475 items
Iteration 9: 9,083 rows | 1,268 users | 1,473 items
Iteration 10: 9,079 rows | 1,266 users | 1,473 items
Iteration 11: 9,079 rows | 1,266 users | 1,473 items

Selected dataset
Core: 3
Interactions: 9079
Users: 1266
Items: 1473
Minimum interactions per user: 3
Minimum interactions per item: 3


In [23]:
SELECTED_K = 3

selected_interactions = iterative_k_core(
    interactions,
    user_k=SELECTED_K,
    item_k=SELECTED_K,
    verbose=True
)

print("\nSelected dataset")
print("Core:", SELECTED_K)
print("Interactions:", len(selected_interactions))
print("Users:", selected_interactions["user_id"].nunique())
print("Items:", selected_interactions["item_id"].nunique())

print(
    "Minimum interactions per user:",
    selected_interactions.groupby("user_id").size().min()
)

print(
    "Minimum interactions per item:",
    selected_interactions.groupby("item_id").size().min()
)

# Giữ tương thích với các bước cũ của notebook
interactions_5core = selected_interactions.copy()

Iteration 1: 17,151 rows | 5,961 users | 2,930 items
Iteration 2: 10,307 rows | 1,833 users | 1,698 items
Iteration 3: 9,444 rows | 1,409 users | 1,544 items
Iteration 4: 9,203 rows | 1,314 users | 1,492 items
Iteration 5: 9,138 rows | 1,286 users | 1,483 items
Iteration 6: 9,120 rows | 1,280 users | 1,480 items
Iteration 7: 9,106 rows | 1,275 users | 1,478 items
Iteration 8: 9,095 rows | 1,272 users | 1,475 items
Iteration 9: 9,083 rows | 1,268 users | 1,473 items
Iteration 10: 9,079 rows | 1,266 users | 1,473 items
Iteration 11: 9,079 rows | 1,266 users | 1,473 items

Selected dataset
Core: 3
Interactions: 9079
Users: 1266
Items: 1473
Minimum interactions per user: 3
Minimum interactions per item: 3


## Bước 12 — Chọn dữ liệu làm project

Mặc định dùng 5-core. Không tự ý đổi sang 3-core trong cùng notebook.

Nếu 5-core quá nhỏ, hãy lưu thống kê rồi thảo luận trước khi thay đổi phương pháp.

In [24]:
if interactions_5core.empty:
    raise ValueError(
        "5-core dataset is empty. Stop here and review the filtering strategy."
    )

project_interactions = interactions_5core.copy()

MAX_USERS = 10_000

unique_users = np.array(sorted(project_interactions["user_id"].unique()))

if len(unique_users) > MAX_USERS:
    selected_users = rng.choice(
        unique_users,
        size=MAX_USERS,
        replace=False,
    )
    project_interactions = project_interactions[
        project_interactions["user_id"].isin(selected_users)
    ].copy()
    sampling_note = f"Randomly sampled {MAX_USERS} users with seed {RANDOM_SEED}."
else:
    sampling_note = (
        f"No sampling was needed because only {len(unique_users)} users remained."
    )

project_interactions = project_interactions.sort_values(
    ["user_id", "timestamp", "item_id"]
).reset_index(drop=True)

print(sampling_note)
print("Project interactions:", f"{len(project_interactions):,}")
print("Project users:", f"{project_interactions['user_id'].nunique():,}")
print("Project items:", f"{project_interactions['item_id'].nunique():,}")

No sampling was needed because only 1266 users remained.
Project interactions: 9,079
Project users: 1,266
Project items: 1,473


## Bước 13 — Chia train/validation/test theo thời gian

Với mỗi user:

- Interaction cuối cùng → test.
- Interaction áp chót → validation.
- Các interaction trước đó → train.

Đây là leave-one-out temporal split.

In [25]:
project_interactions = project_interactions.sort_values(
    ["user_id", "timestamp", "item_id"]
).reset_index(drop=True)

reverse_position = (
    project_interactions.groupby("user_id").cumcount(ascending=False)
)

test_df = project_interactions[reverse_position == 0].copy()
valid_df = project_interactions[reverse_position == 1].copy()
train_df = project_interactions[reverse_position >= 2].copy()

train_df["split"] = "train"
valid_df["split"] = "valid"
test_df["split"] = "test"

split_df = pd.concat(
    [train_df, valid_df, test_df],
    ignore_index=True,
).sort_values(["user_id", "timestamp", "item_id"])

print("Train rows:", f"{len(train_df):,}")
print("Validation rows:", f"{len(valid_df):,}")
print("Test rows:", f"{len(test_df):,}")

print("Users in train:", train_df["user_id"].nunique())
print("Users in validation:", valid_df["user_id"].nunique())
print("Users in test:", test_df["user_id"].nunique())

assert valid_df["user_id"].nunique() == project_interactions["user_id"].nunique()
assert test_df["user_id"].nunique() == project_interactions["user_id"].nunique()
assert len(split_df) == len(project_interactions)

display(split_df.head())

Train rows: 6,547
Validation rows: 1,266
Test rows: 1,266
Users in train: 1266
Users in validation: 1266
Users in test: 1266


,user_id,item_id,timestamp,rating,verified_purchase,datetime,split
0,AE22XHMBOBJBXUFCTNYLFMD4UKMA,B00NT0AR7E,1424615980000,5.0,True,2015-02-22 14:39:40+00:00,train
6547,AE22XHMBOBJBXUFCTNYLFMD4UKMA,B00OS9YWJY,1431982094000,4.0,True,2015-05-18 20:48:14+00:00,valid
7813,AE22XHMBOBJBXUFCTNYLFMD4UKMA,B00DT4757A,1471107037000,5.0,True,2016-08-13 16:50:37+00:00,test
1,AE23NZUELB4BYWLHKWJXAS73PTSQ,B088JWX1QF,1603394717771,2.0,False,2020-10-22 19:25:17.771000+00:00,train
6548,AE23NZUELB4BYWLHKWJXAS73PTSQ,B09T2V5SRH,1661558112977,4.0,False,2022-08-26 23:55:12.977000+00:00,valid


## Bước 14 — Lọc metadata theo catalog cuối cùng

In [26]:
final_item_ids = set(project_interactions["item_id"].unique())

project_items = items_clean[
    items_clean["item_id"].isin(final_item_ids)
].copy()

print("Final interaction items:", len(final_item_ids))
print("Final metadata rows:", len(project_items))
print(
    "Metadata coverage:",
    round(len(project_items) / len(final_item_ids) * 100, 2),
    "%",
)

assert set(final_item_ids).issubset(set(project_items["item_id"]))

Final interaction items: 1473
Final metadata rows: 1473
Metadata coverage: 100.0 %


## Bước 15 — Lưu file processed

In [27]:
project_interactions.to_parquet(
    PROCESSED_DIR / "interactions_full.parquet",
    index=False,
)
train_df.to_parquet(
    PROCESSED_DIR / "train.parquet",
    index=False,
)
valid_df.to_parquet(
    PROCESSED_DIR / "valid.parquet",
    index=False,
)
test_df.to_parquet(
    PROCESSED_DIR / "test.parquet",
    index=False,
)
split_df.to_parquet(
    PROCESSED_DIR / "interactions_with_split.parquet",
    index=False,
)
project_items.to_parquet(
    PROCESSED_DIR / "items.parquet",
    index=False,
)

print("Saved processed files to:", PROCESSED_DIR)

Saved processed files to: /content/drive/MyDrive/llm-next-item-recommendation/data/processed


## Bước 16 — Tạo file `.inter` cho RecBole

RecBole yêu cầu tên cột kèm kiểu dữ liệu.

In [28]:
recbole_interactions = project_interactions[
    ["user_id", "item_id", "timestamp"]
].copy()

recbole_interactions.columns = [
    "user_id:token",
    "item_id:token",
    "timestamp:float",
]

recbole_file = RECBOLE_DIR / "all_beauty.inter"

recbole_interactions.to_csv(
    recbole_file,
    sep="\t",
    index=False,
)

print("RecBole file:", recbole_file)
print("Rows:", len(recbole_interactions))
display(recbole_interactions.head())

RecBole file: /content/drive/MyDrive/llm-next-item-recommendation/data/recbole/all_beauty/all_beauty.inter
Rows: 9079


,user_id:token,item_id:token,timestamp:float
0,AE22XHMBOBJBXUFCTNYLFMD4UKMA,B00NT0AR7E,1424615980000
1,AE22XHMBOBJBXUFCTNYLFMD4UKMA,B00OS9YWJY,1431982094000
2,AE22XHMBOBJBXUFCTNYLFMD4UKMA,B00DT4757A,1471107037000
3,AE23NZUELB4BYWLHKWJXAS73PTSQ,B088JWX1QF,1603394717771
4,AE23NZUELB4BYWLHKWJXAS73PTSQ,B09T2V5SRH,1661558112977


## Bước 17 — Tạo bảng thống kê cuối ngày

In [29]:
user_lengths = project_interactions.groupby("user_id").size()
item_lengths = project_interactions.groupby("item_id").size()

summary = {
    "dataset": "Amazon Reviews 2023 - All Beauty",
    "random_seed": RANDOM_SEED,
    "sampling_note": sampling_note,
    "interactions": int(len(project_interactions)),
    "users": int(project_interactions["user_id"].nunique()),
    "items": int(project_interactions["item_id"].nunique()),
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(valid_df)),
    "test_rows": int(len(test_df)),
    "min_user_interactions": int(user_lengths.min()),
    "median_user_interactions": float(user_lengths.median()),
    "max_user_interactions": int(user_lengths.max()),
    "min_item_interactions": int(item_lengths.min()),
    "median_item_interactions": float(item_lengths.median()),
    "max_item_interactions": int(item_lengths.max()),
    "earliest_datetime": str(project_interactions["datetime"].min()),
    "latest_datetime": str(project_interactions["datetime"].max()),
    "metadata_items": int(len(project_items)),
    "metadata_coverage_pct": round(
        len(project_items)
        / project_interactions["item_id"].nunique()
        * 100,
        2,
    ),
}

with open(
    OUTPUT_DIR / "day1_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

summary_table = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in summary.items()]
)
summary_table.to_csv(
    OUTPUT_DIR / "day1_summary.csv",
    index=False,
)

display(summary_table)

,metric,value
0,dataset,Amazon Reviews 2023 - All Beauty
1,random_seed,42
2,sampling_note,No sampling was needed because only 1266 users...
3,interactions,9079
4,users,1266
5,items,1473
6,train_rows,6547
7,validation_rows,1266
8,test_rows,1266
9,min_user_interactions,3


## Bước 18 — Lưu phiên bản thư viện

In [30]:
package_names = [
    "pandas",
    "numpy",
    "pyarrow",
    "huggingface_hub",
]

versions = {}
for package in package_names:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "not installed"

with open(
    OUTPUT_DIR / "day1_package_versions.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(versions, file, indent=2)

print(versions)

{'pandas': '2.2.2', 'numpy': '2.0.2', 'pyarrow': '24.0.0', 'huggingface_hub': '1.21.0'}


# Checklist hoàn thành Ngày 1

Kiểm tra Google Drive có các file:

- `data/processed/interactions_full.parquet`
- `data/processed/train.parquet`
- `data/processed/valid.parquet`
- `data/processed/test.parquet`
- `data/processed/interactions_with_split.parquet`
- `data/processed/items.parquet`
- `data/recbole/all_beauty/all_beauty.inter`
- `outputs/day1_data_audit.csv`
- `outputs/day1_summary.csv`
- `outputs/day1_summary.json`
- `outputs/day1_package_versions.json`

**Chưa làm trong Ngày 1:**

- Không train mô hình.
- Không cài hoặc chạy LLM.
- Không chọn alpha.
- Không tính Recall/NDCG/MRR.